# Day 3: Data Cleaning & Preprocessing 🧹

**Date:** March 12, 2026  
**Topic:** Missing values, Duplicates, Outliers, Data Types  
**Duration:** 3-4 hours  

## 🎯 Learning Objectives

By the end of this notebook, you will:
1. ✅ Handle missing values effectively
2. ✅ Detect and remove duplicates
3. ✅ Identify and handle outliers
4. ✅ Convert data types correctly
5. ✅ Clean text data
6. ✅ Standardize and normalize data
7. ✅ Create a complete data cleaning pipeline

---

## 📚 Resources for Today

### Documentation
- [Pandas Missing Data](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [Data Cleaning Best Practices](https://www.kaggle.com/docs/datasets)

### Videos (Watch First!)
- [Data Cleaning Basics](https://www.youtube.com/watch?v=video_id) - 25 min ⭐
- [Pandas Error Handling](https://www.youtube.com/watch?v=video_id) - 20 min

### Important Notes:
⚠️ **Data Cleaning Reality:**
- 80% of ML work is data cleaning & preparation
- Cleaning is the most time-consuming step
- Different problems need different solutions
- Document your decisions!
- Never delete data without backup

---

## 🔧 Setup: Install Required Libraries

In [ ]:
# Install required packages
!pip install pandas numpy matplotlib seaborn scipy -q

print("✅ All packages installed successfully!")

## 📦 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print(f"📅 Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n✅ All imports successful!")

---

# Part 1: Understanding Data Quality Issues 📊

## Common Data Quality Issues

| Issue | Cause | Solution |
|-------|-------|----------|
| Missing Values | Collection errors, database issues | Impute or remove |
| Duplicates | Data merging, collection errors | Remove or aggregate |
| Outliers | Measurement errors, anomalies | Cap, remove, or investigate |
| Wrong Data Types | Collection or parsing errors | Convert |
| Inconsistent Formats | Manual entry, system differences | Standardize |
| Text Issues | Whitespace, case, special chars | Clean and normalize |
| Scaled Values | Different measurement units | Standardize |

---

## Exercise 1.1: Create Sample Messy Dataset

Let's create a realistic messy dataset to work with.

In [ ]:
# Create intentionally messy dataset
np.random.seed(42)

messy_data = {
    'Name': ['John Doe', 'jane smith', ' Alex Johnson ', 'Bob_Brown', None, 'Charlie Davis', 'charlie davis', 'Diana Evans'],
    'Email': ['john@example.com', 'jane@example.com', 'alex@example.com', 'bob@example.com', 'charlie@example.com', None, 'charlie@example.com', 'diana@example.com'],
    'Age': [25, 30, np.nan, 35, 28, 45, 28, 29],
    'Salary': ['50000', '60000', 75000, '80,000', 90000, 'N/A', 85000, 72000],
    'Department': ['Sales', 'IT', 'Sales', 'HR', 'IT', 'IT', 'IT', 'Sales'],
    'Date_Joined': ['2020-01-15', '2019-03-22', '2020/05/10', '2019-11-30', '2020-02-14', None, '2020-02-14', '2021-06-01'],
    'Performance_Score': [8.5, 7.2, 9.1, 7.8, 8.9, 10.2, 8.9, 8.0]  # 10.2 is outlier
}

df_messy = pd.DataFrame(messy_data)
print("Original Messy Dataset:")
print(df_messy)
print("\nData Info:")
print(df_messy.info())
print("\nMissing Values:")
print(df_messy.isnull().sum())

---

# Part 2: Handling Missing Values 🚫

## Exercise 2.1: Identify Missing Values

**TASK:** Analyze missing values in the dataset.

In [ ]:
# SOLUTION: Analyze missing values
print("Missing Values Analysis:")
print("="*50)

# Count missing values
missing_count = df_messy.isnull().sum()
missing_percent = (df_messy.isnull().sum() / len(df_messy)) * 100

missing_df = pd.DataFrame({
    'Column': missing_count.index,
    'Missing_Count': missing_count.values,
    'Missing_Percent': missing_percent.values
})

print(missing_df)

# Identify rows with any missing values
print("\nRows with missing values:")
print(df_messy[df_messy.isnull().any(axis=1)])

# Visualize missing values
fig, ax = plt.subplots(figsize=(10, 4))
missing_df.set_index('Column')['Missing_Percent'].plot(kind='bar', ax=ax, color='skyblue')
ax.set_title('Missing Values Percentage by Column')
ax.set_ylabel('Percentage (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Exercise 2.2: Imputation Strategies

**TASK:** Implement different imputation methods for missing values.

In [ ]:
# SOLUTION: Different imputation strategies
df_clean = df_messy.copy()

print("Imputation Strategies:")
print("="*50)

# Strategy 1: Drop rows with missing values
df_dropped = df_messy.dropna()
print(f"\n1. Drop rows with NA:")
print(f"   Original rows: {len(df_messy)}, After dropping: {len(df_dropped)}")

# Strategy 2: Drop columns with too many missing values
threshold = 0.5  # Drop columns with >50% missing
df_dropped_cols = df_messy.dropna(thresh=len(df_messy) * threshold, axis=1)
print(f"\n2. Drop columns with >{threshold*100}% missing: {df_dropped_cols.columns.tolist()}")

# Strategy 3: Fill with mean (for numeric)
df_filled_mean = df_messy.copy()
df_filled_mean['Age'] = df_filled_mean['Age'].fillna(df_filled_mean['Age'].mean())
print(f"\n3. Fill Age with mean:")
print(f"   Age mean: {df_filled_mean['Age'].mean():.1f}")
print(f"   Original NA count: {df_messy['Age'].isnull().sum()}, After fill: {df_filled_mean['Age'].isnull().sum()}")

# Strategy 4: Fill with median (more robust)
df_filled_median = df_messy.copy()
df_filled_median['Age'] = df_filled_median['Age'].fillna(df_filled_median['Age'].median())
print(f"\n4. Fill Age with median: {df_filled_median['Age'].median()}")

# Strategy 5: Fill with forward/backward fill (time series)
df_ffill = df_messy.copy()
df_ffill['Performance_Score'] = df_ffill['Performance_Score'].fillna(method='ffill')  # Forward fill
print(f"\n5. Forward fill (for time series)")

# Strategy 6: Fill with a specific value
df_filled_value = df_messy.copy()
df_filled_value['Date_Joined'] = df_filled_value['Date_Joined'].fillna('Unknown')
print(f"\n6. Fill with specific value ('Unknown' for dates)")

# Strategy 7: Interpolate (for continuous data)
df_interpolate = df_messy.copy()
df_interpolate['Age'] = df_interpolate['Age'].interpolate(method='linear')
print(f"\n7. Interpolate (linear) Age")

print("\nResult after filling Age with median:")
print(df_filled_median[['Name', 'Age', 'Salary']].head(6))

---

# Part 3: Detecting and Removing Duplicates 🔄

## Exercise 3.1: Find and Remove Duplicates

**TASK:** Identify and handle duplicate records.

In [ ]:
# SOLUTION: Handle duplicates
print("Duplicate Analysis:")
print("="*50)

# Find duplicate rows
print(f"\nTotal rows: {len(df_messy)}")
print(f"Exact duplicates: {df_messy.duplicated().sum()}")

# Find duplicates based on specific columns
dup_email = df_messy.duplicated(subset=['Email'], keep=False)
dup_name = df_messy.duplicated(subset=['Name'], keep=False)

print(f"Duplicate emails: {df_messy[dup_email]['Email'].value_counts().sum() - dup_email.value_counts()[True]}")
print(f"Duplicate names: {df_messy[dup_name]['Name'].value_counts().sum() - dup_name.value_counts()[True]}")

# Show duplicate records
print("\nDuplicate name records:")
print(df_messy[df_messy.duplicated(subset=['Name'], keep=False)].sort_values('Name'))

# Remove duplicates
df_no_dup = df_messy.drop_duplicates(keep='first')
print(f"\nAfter removing exact duplicates:")
print(f"  Original: {len(df_messy)} rows, After: {len(df_no_dup)} rows")

# Remove duplicates by specific column
df_dup_email = df_messy.drop_duplicates(subset=['Email'], keep='first')
print(f"\nAfter removing by Email:")
print(f"  Original: {len(df_messy)} rows, After: {len(df_dup_email)} rows")

---

# Part 4: Outlier Detection & Handling 📈

## Exercise 4.1: Identify Outliers

**TASK:** Detect outliers using multiple methods.

In [ ]:
# SOLUTION: Outlier detection methods
print("Outlier Detection Methods:")
print("="*50)

# Prepare data (fill missing Age)
df_for_outliers = df_messy.copy()
df_for_outliers['Age'] = df_for_outliers['Age'].fillna(df_for_outliers['Age'].median())

# Method 1: Standard Deviation (Z-score)
print("\n1. Z-SCORE Method (|z| > 3):")
z_scores = np.abs((df_for_outliers['Performance_Score'] - df_for_outliers['Performance_Score'].mean()) / df_for_outliers['Performance_Score'].std())
outliers_z = z_scores > 3
print(f"   Outliers found: {outliers_z.sum()}")
print(f"   Values: {df_for_outliers[outliers_z]['Performance_Score'].values}")

# Method 2: IQR (Interquartile Range)
print("\n2. IQR Method:")
Q1 = df_for_outliers['Performance_Score'].quantile(0.25)
Q3 = df_for_outliers['Performance_Score'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"   Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")
print(f"   Lower bound: {lower_bound:.2f}, Upper bound: {upper_bound:.2f}")

outliers_iqr = (df_for_outliers['Performance_Score'] < lower_bound) | (df_for_outliers['Performance_Score'] > upper_bound)
print(f"   Outliers found: {outliers_iqr.sum()}")
print(f"   Values: {df_for_outliers[outliers_iqr]['Performance_Score'].values}")

# Method 3: Percentile Method
print("\n3. Percentile Method (< 1st or > 99th percentile):")
p1 = df_for_outliers['Performance_Score'].quantile(0.01)
p99 = df_for_outliers['Performance_Score'].quantile(0.99)
outliers_percentile = (df_for_outliers['Performance_Score'] < p1) | (df_for_outliers['Performance_Score'] > p99)
print(f"   1st percentile: {p1:.2f}, 99th percentile: {p99:.2f}")
print(f"   Outliers found: {outliers_percentile.sum()}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Box plot
axes[0].boxplot(df_for_outliers['Performance_Score'])
axes[0].set_title('Box Plot - Performance Score')
axes[0].set_ylabel('Score')

# Distribution
axes[1].hist(df_for_outliers['Performance_Score'], bins=10, edgecolor='black', alpha=0.7)
axes[1].axvline(df_for_outliers['Performance_Score'].mean(), color='red', label='Mean')
axes[1].axvline(df_for_outliers['Performance_Score'].median(), color='green', label='Median')
axes[1].set_title('Distribution - Performance Score')
axes[1].set_xlabel('Score')
axes[1].legend()

plt.tight_layout()
plt.show()

## Exercise 4.2: Handle Outliers

**TASK:** Apply different strategies for handling outliers.

In [ ]:
# SOLUTION: Handling outliers
print("Outlier Handling Methods:")
print("="*50)

df_outlier_handling = df_for_outliers.copy()

# Method 1: Remove outliers
df_removed = df_outlier_handling[(df_outlier_handling['Performance_Score'] >= lower_bound) & 
                                  (df_outlier_handling['Performance_Score'] <= upper_bound)]
print(f"\n1. REMOVE Outliers:")
print(f"   Before: {len(df_outlier_handling)} rows, After: {len(df_removed)} rows")
print(f"   Removed: {df_outlier_handling[outliers_iqr]['Performance_Score'].values}")

# Method 2: Cap/Clip outliers (winsorize)
df_capped = df_outlier_handling.copy()
df_capped['Performance_Score'] = df_capped['Performance_Score'].clip(lower=lower_bound, upper=upper_bound)
print(f"\n2. CAP Outliers:")
print(f"   Original max: 10.2, Capped max: {df_capped['Performance_Score'].max()}")
print(f"   Original: {df_outlier_handling['Performance_Score'].values}")
print(f"   After cap: {df_capped['Performance_Score'].values}")

# Method 3: Replace with median
df_median_replace = df_outlier_handling.copy()
median_val = df_median_replace.loc[~outliers_iqr, 'Performance_Score'].median()
df_median_replace.loc[outliers_iqr, 'Performance_Score'] = median_val
print(f"\n3. REPLACE with Median:")
print(f"   Replaced outlier 10.2 with median: {median_val}")

# Method 4: Transform (log, sqrt)
print(f"\n4. TRANSFORM Data (Log transformation):")
df_transformed = df_outlier_handling.copy()
df_transformed['Performance_Score_Log'] = np.log(df_transformed['Performance_Score'])
print(f"   Original: {df_outlier_handling['Performance_Score'].values}")
print(f"   After log: {df_transformed['Performance_Score_Log'].values.round(2)}")

print(f"\n✅ Choose method based on:")
print(f"   - Cause of outlier (measurement error vs. real anomaly)")
print(f"   - Business context")
print(f"   - Impact on analysis")

---

# Part 5: Data Type Conversion 🔤

## Exercise 5.1: Fix Data Types

**TASK:** Convert columns to appropriate data types.

In [ ]:
# SOLUTION: Data type conversion
print("Data Type Conversion:")
print("="*50)

df_typed = df_messy.copy()

# 1. Fix Age (ensure numeric)
print("\n1. Age column:")
print(f"   Before: {df_typed['Age'].dtype}")
df_typed['Age'] = pd.to_numeric(df_typed['Age'], errors='coerce')
print(f"   After: {df_typed['Age'].dtype}")
print(f"   Missing after conversion: {df_typed['Age'].isnull().sum()}")

# 2. Fix Salary (remove $ and commas, convert to numeric)
print("\n2. Salary column:")
print(f"   Original values: {df_typed['Salary'].value_counts().to_dict()}")
df_typed['Salary'] = df_typed['Salary'].astype(str).str.replace(',', '').str.replace('$', '')
df_typed['Salary'] = pd.to_numeric(df_typed['Salary'], errors='coerce')
print(f"   After conversion: {df_typed['Salary'].dtype}")
print(f"   Sample values: {df_typed['Salary'].dropna().values.astype(int)}")

# 3. Fix Date (parse different formats)
print("\n3. Date_Joined column:")
print(f"   Before: {df_typed['Date_Joined'].dtype}")
df_typed['Date_Joined'] = pd.to_datetime(df_typed['Date_Joined'], errors='coerce')
print(f"   After: {df_typed['Date_Joined'].dtype}")
print(f"   Sample values: {df_typed['Date_Joined'].dropna().values}")

# 4. Convert to appropriate types
df_typed['Department'] = df_typed['Department'].astype('category')
print(f"\n4. Department as category: {df_typed['Department'].dtype}")

print(f"\nFinal data types:")
print(df_typed.dtypes)

---

# Part 6: Text Data Cleaning 📝

## Exercise 6.1: Clean Text Data

**TASK:** Standardize and clean text fields.

In [ ]:
# SOLUTION: Text cleaning
print("Text Data Cleaning:")
print("="*50)

df_text = df_messy.copy()

# Show original values
print("\nOriginal Names:")
print(df_text['Name'].values)

# Method 1: Strip whitespace
df_text['Name'] = df_text['Name'].str.strip()
print("\n1. After stripping whitespace:")
print(df_text['Name'].values)

# Method 2: Standardize case
df_text['Name'] = df_text['Name'].str.title()
print("\n2. After title case:")
print(df_text['Name'].values)

# Method 3: Remove special characters
df_text['Name'] = df_text['Name'].str.replace('_', ' ')
print("\n3. After removing underscores:")
print(df_text['Name'].values)

# Method 4: Remove duplicates based on cleaned text
print("\n4. Handling near-duplicates (case-insensitive):")
print(f"   Before dedup: {len(df_text)} rows")
df_text_unique = df_text.drop_duplicates(subset=['Name'], keep='first')
print(f"   After dedup: {len(df_text_unique)} rows")

# Method 5: Remove rows with missing names
print("\n5. Remove null names:")
print(f"   Before: {len(df_text_unique)} rows")
df_text_clean = df_text_unique[df_text_unique['Name'].notna()]
print(f"   After: {len(df_text_clean)} rows")

print(f"\n✅ Final cleaned names:")
print(df_text_clean['Name'].values)

---

# Part 7: Standardization & Normalization 📏

## Exercise 7.1: Standardize & Normalize Numerical Data

**TASK:** Apply scaling techniques to numerical features.

In [ ]:
# SOLUTION: Standardization and Normalization
print("Standardization & Normalization:")
print("="*50)

df_scale = df_for_outliers.copy()

# Create numeric columns for scaling
numeric_cols = ['Age', 'Performance_Score']

print("\nOriginal values:")
print(df_scale[numeric_cols].describe())

# Method 1: Min-Max Normalization (0-1)
print("\n1. MIN-MAX Normalization (0-1 range):")
df_minmax = df_scale.copy()
for col in numeric_cols:
    min_val = df_minmax[col].min()
    max_val = df_minmax[col].max()
    df_minmax[f'{col}_MinMax'] = (df_minmax[col] - min_val) / (max_val - min_val)

print("Age normalized:")
print(f"  Original range: {df_scale['Age'].min():.1f} - {df_scale['Age'].max():.1f}")
print(f"  New range: {df_minmax['Age_MinMax'].min():.2f} - {df_minmax['Age_MinMax'].max():.2f}")
print(df_minmax[['Age', 'Age_MinMax']].dropna())

# Method 2: Standardization (Z-score)
print("\n2. STANDARDIZATION (Z-score):")
df_zscore = df_scale.copy()
for col in numeric_cols:
    mean = df_zscore[col].mean()
    std = df_zscore[col].std()
    df_zscore[f'{col}_ZScore'] = (df_zscore[col] - mean) / std

print("Age standardized:")
print(f"  Original: mean={df_scale['Age'].mean():.1f}, std={df_scale['Age'].std():.1f}")
print(f"  Standardized: mean={df_zscore['Age_ZScore'].mean():.2f}, std={df_zscore['Age_ZScore'].std():.2f}")
print(df_zscore[['Age', 'Age_ZScore']].dropna())

# Method 3: Robust Scaling (using median and IQR)
print("\n3. ROBUST Scaling (median-based):")
df_robust = df_scale.copy()
for col in numeric_cols:
    median = df_robust[col].median()
    Q1 = df_robust[col].quantile(0.25)
    Q3 = df_robust[col].quantile(0.75)
    IQR = Q3 - Q1
    df_robust[f'{col}_Robust'] = (df_robust[col] - median) / IQR

print("Performance score robust scaled:")
print(df_robust[['Performance_Score', 'Performance_Score_Robust']])

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(df_scale['Age'], bins=10, alpha=0.7, edgecolor='black')
axes[0, 0].set_title('Original Age')

axes[0, 1].hist(df_minmax['Age_MinMax'].dropna(), bins=10, alpha=0.7, color='green', edgecolor='black')
axes[0, 1].set_title('Min-Max Normalized')

axes[1, 0].hist(df_zscore['Age_ZScore'].dropna(), bins=10, alpha=0.7, color='orange', edgecolor='black')
axes[1, 0].set_title('Z-Score Standardized')

axes[1, 1].hist(df_robust['Age_Robust'].dropna(), bins=10, alpha=0.7, color='red', edgecolor='black')
axes[1, 1].set_title('Robust Scaled')

plt.tight_layout()
plt.show()

---

# 🎯 Project: Complete Data Cleaning Pipeline

**Your Mission:** Build a complete cleaning pipeline that:
1. Loads the messy data
2. Analyzes data quality
3. Handles missing values
4. Removes duplicates
5. Detects and handles outliers
6. Fixes data types
7. Cleans text data
8. Standardizes numerical values
9. Validates cleaned data
10. Saves clean data

In [ ]:
# SOLUTION: Complete Data Cleaning Pipeline

class DataCleaningPipeline:
    """Complete data cleaning pipeline."""
    
    def __init__(self, df):
        self.original_df = df.copy()
        self.df = df.copy()
        self.report = {}
    
    def analyze_quality(self):
        """Analyze data quality."""
        print("\n" + "="*60)
        print("STEP 1: ANALYZING DATA QUALITY")
        print("="*60)
        
        self.report['shape_before'] = self.df.shape
        self.report['missing_values'] = self.df.isnull().sum().to_dict()
        
        print(f"Shape: {self.df.shape}")
        print(f"\nMissing values:\n{pd.Series(self.report['missing_values'])}")
        print(f"\nData types:\n{self.df.dtypes}")
        print(f"\nDuplicates: {self.df.duplicated().sum()}")
        
        return self
    
    def fix_missing(self):
        """Handle missing values."""
        print("\n" + "="*60)
        print("STEP 2: HANDLING MISSING VALUES")
        print("="*60)
        
        # Fill Age with median
        if 'Age' in self.df.columns:
            self.df['Age'] = self.df['Age'].fillna(self.df['Age'].median())
            print("✅ Filled Age with median")
        
        # Fill Date_Joined with mode
        if 'Date_Joined' in self.df.columns:
            self.df['Date_Joined'] = self.df['Date_Joined'].fillna('Unknown')
            print("✅ Filled Date_Joined with 'Unknown'")
        
        # Drop rows with critical missing values
        if 'Name' in self.df.columns:
            before = len(self.df)
            self.df = self.df[self.df['Name'].notna()]
            print(f"✅ Dropped {before - len(self.df)} rows with missing Name")
        
        return self
    
    def remove_duplicates(self):
        """Remove duplicate records."""
        print("\n" + "="*60)
        print("STEP 3: REMOVING DUPLICATES")
        print("="*60)
        
        before = len(self.df)
        self.df = self.df.drop_duplicates(keep='first')
        print(f"✅ Removed {before - len(self.df)} duplicate rows")
        
        # Handle case-insensitive duplicates in Name
        if 'Name' in self.df.columns:
            before = len(self.df)
            self.df['Name'] = self.df['Name'].str.strip().str.title()
            self.df = self.df.drop_duplicates(subset=['Name'], keep='first')
            print(f"✅ Removed {before - len(self.df)} case-insensitive name duplicates")
        
        return self
    
    def fix_datatypes(self):
        """Fix data types."""
        print("\n" + "="*60)
        print("STEP 4: FIXING DATA TYPES")
        print("="*60)
        
        # Fix Age
        if 'Age' in self.df.columns:
            self.df['Age'] = pd.to_numeric(self.df['Age'], errors='coerce')
            print("✅ Age -> numeric")
        
        # Fix Salary
        if 'Salary' in self.df.columns:
            self.df['Salary'] = self.df['Salary'].astype(str).str.replace('[^0-9]', '', regex=True)
            self.df['Salary'] = pd.to_numeric(self.df['Salary'], errors='coerce')
            print("✅ Salary -> numeric")
        
        # Fix Date
        if 'Date_Joined' in self.df.columns:
            self.df['Date_Joined'] = pd.to_datetime(self.df['Date_Joined'], errors='coerce')
            print("✅ Date_Joined -> datetime")
        
        # Fix Department as category
        if 'Department' in self.df.columns:
            self.df['Department'] = self.df['Department'].astype('category')
            print("✅ Department -> category")
        
        return self
    
    def handle_outliers(self, columns=['Performance_Score']):
        """Handle outliers using IQR method."""
        print("\n" + "="*60)
        print("STEP 5: HANDLING OUTLIERS")
        print("="*60)
        
        for col in columns:
            if col in self.df.columns:
                Q1 = self.df[col].quantile(0.25)
                Q3 = self.df[col].quantile(0.75)
                IQR = Q3 - Q1
                lower = Q1 - 1.5 * IQR
                upper = Q3 + 1.5 * IQR
                
                outliers = ((self.df[col] < lower) | (self.df[col] > upper)).sum()
                
                # Cap outliers
                self.df[col] = self.df[col].clip(lower=lower, upper=upper)
                print(f"✅ {col}: Capped {outliers} outliers (bounds: {lower:.2f} - {upper:.2f})")
        
        return self
    
    def validate(self):
        """Validate cleaned data."""
        print("\n" + "="*60)
        print("STEP 6: VALIDATING CLEANED DATA")
        print("="*60)
        
        print(f"Shape: {self.df.shape}")
        print(f"Missing values: {self.df.isnull().sum().sum()}")
        print(f"Duplicates: {self.df.duplicated().sum()}")
        print(f"\nData types:\n{self.df.dtypes}")
        
        self.report['shape_after'] = self.df.shape
        self.report['rows_removed'] = self.report['shape_before'][0] - self.df.shape[0]
        
        return self
    
    def get_clean_data(self):
        """Return cleaned data."""
        print("\n" + "="*60)
        print("✅ CLEANING COMPLETE!")
        print("="*60)
        print(f"Rows removed: {self.report['rows_removed']}")
        print(f"Original shape: {self.report['shape_before']}")
        print(f"Cleaned shape: {self.report['shape_after']}")
        
        return self.df


# Run the pipeline
cleaner = DataCleaningPipeline(df_messy)
df_final = (cleaner
    .analyze_quality()
    .fix_missing()
    .remove_duplicates()
    .fix_datatypes()
    .handle_outliers()
    .validate()
    .get_clean_data())

print("\nFinal Clean Data:")
print(df_final)

## Template: Reusable DataCleaningPipeline Class

In [ ]:
# Already provided above - can be imported and reused
print("✅ DataCleaningPipeline class is ready to use!")
print("\nUsage example:")
print("""cleaner = DataCleaningPipeline(your_df)
clean_df = (cleaner
    .analyze_quality()
    .fix_missing()
    .remove_duplicates()
    .fix_datatypes()
    .handle_outliers()
    .validate()
    .get_clean_data())""")

---

# 📝 Day 3 Summary & Checklist

## What You Learned Today:
- ✅ Identifying data quality issues
- ✅ Handling missing values (multiple strategies)
- ✅ Detecting and removing duplicates
- ✅ Identifying and handling outliers
- ✅ Converting data types
- ✅ Cleaning text data
- ✅ Standardizing and normalizing values
- ✅ Building a complete cleaning pipeline

## Self-Assessment:
Rate yourself (1-5) on:
- [ ] Missing value handling: ___/5
- [ ] Duplicate detection: ___/5
- [ ] Outlier handling: ___/5
- [ ] Data type conversion: ___/5
- [ ] Pipeline design: ___/5

## Practice Projects:
1. **Clean Kaggle dataset** (any dataset)
2. **Cryptocurrency data** (from Day 2 collection + clean it)
3. **Student records** (simulate messy education data)
4. **E-commerce data** (product/order data)
5. **Weather data** (historical weather with issues)

## Code to Remember:
```python
# Missing values
df['col'].fillna(df['col'].mean())  # Fill with mean
df['col'].fillna(method='ffill')    # Forward fill
df.dropna()                          # Drop rows

# Duplicates
df.drop_duplicates()
df.drop_duplicates(subset=['col'], keep='first')

# Outliers (IQR method)
Q1 = df['col'].quantile(0.25)
Q3 = df['col'].quantile(0.75)
df['col'].clip(lower=Q1-1.5*(Q3-Q1), upper=Q3+1.5*(Q3-Q1))

# Data types
df['col'] = pd.to_numeric(df['col'], errors='coerce')
df['col'] = pd.to_datetime(df['col'])

# Text cleaning
df['col'].str.strip().str.title().str.replace('_', ' ')

# Standardization
(df['col'] - df['col'].mean()) / df['col'].std()  # Z-score
(df['col'] - df['col'].min()) / (df['col'].max() - df['col'].min())  # MinMax
```

---

## 🎯 What's Next?

**Tomorrow (Day 4): Exploratory Data Analysis (EDA)**
- Descriptive statistics
- Data visualization
- Feature relationships
- Correlation analysis
- Distribution analysis

**Preparation:**
1. Review matplotlib/seaborn documentation
2. Prepare a cleaned dataset (use today's pipeline)
3. Watch: [Matplotlib Tutorial](https://www.youtube.com/watch?v=...)

---

# 💪 Excellent Work on Day 3!

Data cleaning is the foundation of good ML. Remember:
1. Always save the original data
2. Document your decisions
3. Validate after each step
4. Create reusable pipelines
5. Test edge cases

**Tasks before tomorrow:**
- [ ] Complete all practice exercises
- [ ] Clean one real dataset using the pipeline
- [ ] Create data quality report
- [ ] Update PROGRESS.md
- [ ] Commit code to Git

**See you for Day 4 - Exploratory Data Analysis!** 📊✨